# Lab 03: End-to-End RAG Evaluation

## Objectives
- Build a simple Retrieval-Augmented Generation (RAG) system
- Evaluate RAG quality using Ragas metrics
- Analyze retrieval vs. generation components separately
- Identify and categorize failure modes

## Why RAG Evaluation Matters
- RAG systems combine retrieval and generation quality
- Failures can occur at retrieval, ranking, or generation stages
- Component analysis helps prioritize improvements
- Ragas provides LLM-based metrics without reference answers

In [ ]:
# Install dependencies
!pip install ragas langchain faiss-cpu openai tiktoken pandas numpy -q

import os
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

print("Dependencies installed successfully!")

## Step 1: Build Simple RAG System

Create a minimal RAG pipeline with vector store and QA chain.

In [ ]:
@dataclass
class Document:
    content: str
    metadata: Dict

class SimpleVectorStore:
    """Minimal vector store using embeddings and cosine similarity"""
    
    def __init__(self):
        self.documents = []
        self.embeddings = []
    
    def add_documents(self, docs: List[Document]):
        """Add documents to the store (simulated embeddings)"""
        self.documents.extend(docs)
        # Simulate embeddings: hash-based for consistency
        for doc in docs:
            # Deterministic "embedding" based on content
            embedding = self._get_embedding(doc.content)
            self.embeddings.append(embedding)
    
    def _get_embedding(self, text: str) -> np.ndarray:
        """Generate deterministic 'embedding' for demo"""
        import hashlib
        hash_val = int(hashlib.md5(text.encode()).hexdigest(), 16)
        np.random.seed(hash_val % (2**32))
        return np.random.randn(10)
    
    def retrieve(self, query: str, k: int = 3) -> List[Document]:
        """Retrieve top-k documents similar to query"""
        query_emb = self._get_embedding(query)
        scores = []
        
        for emb in self.embeddings:
            # Cosine similarity
            sim = np.dot(query_emb, emb) / (np.linalg.norm(query_emb) * np.linalg.norm(emb) + 1e-8)
            scores.append(sim)
        
        top_indices = np.argsort(scores)[-k:][::-1]
        return [self.documents[i] for i in top_indices]


class SimpleRAG:
    """Simple RAG system combining retrieval and generation"""
    
    def __init__(self):
        self.vector_store = SimpleVectorStore()
        self.knowledge_base_loaded = False
    
    def load_knowledge_base(self, documents: List[Document]):
        """Load documents into the knowledge base"""
        self.vector_store.add_documents(documents)
        self.knowledge_base_loaded = True
    
    def query(self, question: str, k: int = 3) -> Dict:
        """Execute RAG query: retrieve then generate"""
        # Retrieval phase
        retrieved_docs = self.vector_store.retrieve(question, k=k)
        
        # Generation phase (simulated)
        context = "\n".join([doc.content for doc in retrieved_docs])
        answer = self._generate_answer(question, context)
        
        return {
            "question": question,
            "retrieved_context": context,
            "retrieved_docs": [doc.content for doc in retrieved_docs],
            "answer": answer
        }
    
    def _generate_answer(self, question: str, context: str) -> str:
        """Simulate answer generation"""
        # Simulated LLM response
        if not context:
            return f"I don't have relevant information to answer '{question}'."
        
        # Simple heuristic-based answer generation
        if "python" in question.lower() and "programming" in context.lower():
            return "Python is a versatile programming language used for web development, data analysis, and machine learning."
        elif "machine learning" in question.lower():
            return "Machine learning is a field of AI that enables systems to learn and improve from experience without explicit programming."
        elif "evaluation" in question.lower():
            return "LLM evaluation involves assessing model outputs across multiple dimensions like accuracy, coherence, and safety."
        else:
            return f"Based on the available information: {context[:100]}..."

print("RAG system components implemented")

## Step 2: Create Evaluation Dataset

Build a dataset with questions, ground truth answers, and expected retrieved documents.

In [ ]:
# Create knowledge base documents
knowledge_base = [
    Document(
        content="Python is a high-level, interpreted programming language known for its simple syntax and readability. It supports multiple programming paradigms and has a rich ecosystem of libraries.",
        metadata={"source": "programming_basics", "topic": "python"}
    ),
    Document(
        content="Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. Common approaches include supervised, unsupervised, and reinforcement learning.",
        metadata={"source": "ml_fundamentals", "topic": "machine_learning"}
    ),
    Document(
        content="Deep learning uses neural networks with multiple layers to learn hierarchical representations. It has revolutionized computer vision, natural language processing, and speech recognition.",
        metadata={"source": "ml_advanced", "topic": "deep_learning"}
    ),
    Document(
        content="Natural Language Processing (NLP) focuses on enabling computers to understand, interpret, and generate human language. Techniques include tokenization, parsing, and semantic analysis.",
        metadata={"source": "nlp_intro", "topic": "nlp"}
    ),
    Document(
        content="Evaluation metrics for machine learning models include accuracy, precision, recall, F1-score, and AUC-ROC. Choosing the right metric depends on the task and data distribution.",
        metadata={"source": "ml_evaluation", "topic": "evaluation"}
    ),
    Document(
        content="Large Language Models (LLMs) are transformer-based neural networks trained on vast amounts of text data. They can perform various NLP tasks through prompting and fine-tuning.",
        metadata={"source": "llm_basics", "topic": "llm"}
    ),
    Document(
        content="Retrieval-Augmented Generation (RAG) combines information retrieval with generative models to improve response accuracy and relevance. It addresses hallucination issues in LLMs.",
        metadata={"source": "rag_intro", "topic": "rag"}
    ),
    Document(
        content="Vector embeddings represent text as numerical vectors in a high-dimensional space. They enable similarity-based retrieval and are fundamental to modern RAG systems.",
        metadata={"source": "embeddings", "topic": "embeddings"}
    ),
    Document(
        content="The attention mechanism in transformers allows models to weigh different parts of input differently. It enables parallel processing and better capture of long-range dependencies.",
        metadata={"source": "transformers", "topic": "architecture"}
    ),
    Document(
        content="Fine-tuning involves adapting a pre-trained model to specific downstream tasks with smaller, task-specific datasets. It's more efficient than training from scratch.",
        metadata={"source": "transfer_learning", "topic": "fine_tuning"}
    )
]

# Create evaluation QA pairs
eval_qa_pairs = [
    {
        "question": "What is Python and why is it popular?",
        "ground_truth": "Python is a high-level, interpreted programming language popular for its simple syntax, readability, and rich library ecosystem. It supports multiple programming paradigms.",
        "expected_retrieved_topics": ["python"]
    },
    {
        "question": "Explain machine learning and its main approaches",
        "ground_truth": "Machine learning is a subset of AI that enables systems to learn from experience. Main approaches include supervised learning, unsupervised learning, and reinforcement learning.",
        "expected_retrieved_topics": ["machine_learning"]
    },
    {
        "question": "How does deep learning differ from traditional machine learning?",
        "ground_truth": "Deep learning uses neural networks with multiple layers to learn hierarchical representations, while traditional ML often requires manual feature engineering. Deep learning has excelled in vision and NLP.",
        "expected_retrieved_topics": ["deep_learning", "machine_learning"]
    },
    {
        "question": "What is Natural Language Processing?",
        "ground_truth": "NLP focuses on enabling computers to understand, interpret, and generate human language. Key techniques include tokenization, parsing, and semantic analysis.",
        "expected_retrieved_topics": ["nlp"]
    },
    {
        "question": "How do you evaluate machine learning models?",
        "ground_truth": "Common metrics include accuracy, precision, recall, F1-score, and AUC-ROC. The choice depends on the task and data distribution.",
        "expected_retrieved_topics": ["evaluation"]
    },
    {
        "question": "What are Large Language Models?",
        "ground_truth": "LLMs are transformer-based neural networks trained on vast text data. They can perform various NLP tasks through prompting and fine-tuning.",
        "expected_retrieved_topics": ["llm"]
    },
    {
        "question": "Explain Retrieval-Augmented Generation",
        "ground_truth": "RAG combines information retrieval with generative models to improve response accuracy and reduce hallucination in LLMs.",
        "expected_retrieved_topics": ["rag"]
    },
    {
        "question": "What are vector embeddings and why are they important?",
        "ground_truth": "Vector embeddings represent text as numerical vectors in high-dimensional space, enabling similarity-based retrieval. They're fundamental to modern RAG systems.",
        "expected_retrieved_topics": ["embeddings"]
    },
    {
        "question": "How does the attention mechanism work in transformers?",
        "ground_truth": "Attention allows models to weigh different input parts differently, enabling parallel processing and better capture of long-range dependencies.",
        "expected_retrieved_topics": ["architecture"]
    },
    {
        "question": "What is fine-tuning and when should it be used?",
        "ground_truth": "Fine-tuning adapts pre-trained models to specific tasks with smaller datasets. It's more efficient than training from scratch and effective for transfer learning.",
        "expected_retrieved_topics": ["fine_tuning"]
    }
]

print(f"Created knowledge base with {len(knowledge_base)} documents")
print(f"Created evaluation dataset with {len(eval_qa_pairs)} QA pairs")

## Step 3: Run Ragas Evaluation

Evaluate RAG system using Ragas metrics for faithfulness, context relevancy, and answer relevancy.

In [ ]:
# Initialize RAG system
rag = SimpleRAG()
rag.load_knowledge_base(knowledge_base)

# Run RAG on all QA pairs and collect results
rag_results = []

for qa_pair in eval_qa_pairs:
    rag_output = rag.query(qa_pair["question"], k=3)
    
    # Simulate Ragas metrics
    # In production: faithfulness, context_relevancy, answer_relevancy = ragas.evaluate(...)
    faithfulness = 0.75 + np.random.uniform(-0.1, 0.15)  # LLM-based metric
    context_relevancy = 0.80 + np.random.uniform(-0.1, 0.15)
    answer_relevancy = 0.70 + np.random.uniform(-0.1, 0.15)
    
    rag_results.append({
        "question": qa_pair["question"],
        "ground_truth": qa_pair["ground_truth"],
        "generated_answer": rag_output["answer"],
        "retrieved_context": rag_output["retrieved_context"][:150],
        "faithfulness": np.clip(faithfulness, 0, 1),
        "context_relevancy": np.clip(context_relevancy, 0, 1),
        "answer_relevancy": np.clip(answer_relevancy, 0, 1)
    })

# Convert to DataFrame
df_rag = pd.DataFrame(rag_results)

# Overall metrics
print("=== RAG Evaluation Results ===")
print(f"\nOverall Metrics (across {len(df_rag)} queries):")
print(f"  Faithfulness:       {df_rag['faithfulness'].mean():.3f} ± {df_rag['faithfulness'].std():.3f}")
print(f"  Context Relevancy:  {df_rag['context_relevancy'].mean():.3f} ± {df_rag['context_relevancy'].std():.3f}")
print(f"  Answer Relevancy:   {df_rag['answer_relevancy'].mean():.3f} ± {df_rag['answer_relevancy'].std():.3f}")

# Composite score
composite = (df_rag['faithfulness'] + df_rag['context_relevancy'] + df_rag['answer_relevancy']) / 3
print(f"  Composite Score:    {composite.mean():.3f} (Average of the three metrics)")

print(f"\nSample Results:")
print(df_rag[['question', 'faithfulness', 'context_relevancy', 'answer_relevancy']].head().to_string(index=False))

## Step 4: Component Analysis

Separate analysis of retrieval quality vs. generation quality.

In [ ]:
# Component-level analysis
df_rag['generation_quality'] = (df_rag['faithfulness'] + df_rag['answer_relevancy']) / 2
df_rag['retrieval_quality'] = df_rag['context_relevancy']

# Correlations
print("=== Component Analysis ===")
print(f"\nRetrieval Quality (Context Relevancy):")
print(f"  Mean: {df_rag['retrieval_quality'].mean():.3f}")
print(f"  Min:  {df_rag['retrieval_quality'].min():.3f}")
print(f"  Max:  {df_rag['retrieval_quality'].max():.3f}")

print(f"\nGeneration Quality (Faithfulness + Answer Relevancy):")
print(f"  Mean: {df_rag['generation_quality'].mean():.3f}")
print(f"  Min:  {df_rag['generation_quality'].min():.3f}")
print(f"  Max:  {df_rag['generation_quality'].max():.3f}")

# Identify bottlenecks
retrieval_limited = (df_rag['retrieval_quality'] < 0.7).sum()
generation_limited = (df_rag['generation_quality'] < 0.7).sum()

print(f"\nBottleneck Analysis:")
print(f"  Queries with poor retrieval (< 0.7): {retrieval_limited}/{len(df_rag)}")
print(f"  Queries with poor generation (< 0.7): {generation_limited}/{len(df_rag)}")
print(f"  Bottleneck type: {'Retrieval-limited' if retrieval_limited > generation_limited else 'Generation-limited'}")

# Correlation between components
corr = df_rag['retrieval_quality'].corr(df_rag['generation_quality'])
print(f"\nComponent correlation: {corr:.3f}")
if corr > 0.7:
    print("  → Strong coupling: Retrieval issues significantly impact generation")
elif corr > 0.4:
    print("  → Moderate coupling: Some interaction between components")
else:
    print("  → Weak coupling: Components can fail independently")

## Step 5: Failure Analysis

Identify and categorize failure modes in the RAG pipeline.

In [ ]:
# Categorize failures
failure_categories = []

for idx, row in df_rag.iterrows():
    retrieval = row['retrieval_quality']
    generation = row['generation_quality']
    
    if retrieval >= 0.7 and generation >= 0.7:
        category = "Success"
    elif retrieval < 0.7 and generation >= 0.7:
        category = "Retrieval Failure"
    elif retrieval >= 0.7 and generation < 0.7:
        category = "Generation Failure"
    else:
        category = "Combined Failure"
    
    failure_categories.append(category)

df_rag['failure_category'] = failure_categories

# Failure statistics
print("=== Failure Analysis ===")
failure_counts = df_rag['failure_category'].value_counts()
print("\nFailure Breakdown:")
for category, count in failure_counts.items():
    pct = (count / len(df_rag)) * 100
    print(f"  {category:25s}: {count:2d}/{len(df_rag)} ({pct:5.1f}%)")

# Detailed failure examples
print("\nDetailed Failure Examples:")
for category in ["Retrieval Failure", "Generation Failure", "Combined Failure"]:
    examples = df_rag[df_rag['failure_category'] == category].head(2)
    if len(examples) > 0:
        print(f"\n{category}:")
        for idx, row in examples.iterrows():
            print(f"  Q: {row['question'][:50]}...")
            print(f"  Retrieval: {row['retrieval_quality']:.2f}, Generation: {row['generation_quality']:.2f}")

## Recommendations

Based on the RAG evaluation, here are actionable recommendations:

### For Retrieval Issues
- Improve embedding model quality
- Expand knowledge base coverage
- Implement better document chunking
- Add semantic routing for domain-specific queries

### For Generation Issues
- Fine-tune LLM on domain-specific data
- Implement prompt engineering for consistency
- Add hallucination detection and mitigation
- Use ground truth in-context examples

### System-Level Improvements
- Add query expansion and rewriting
- Implement iterative retrieval (multi-hop)
- Use re-ranking to improve context quality
- Add human feedback loops for continuous improvement

### Measurement Strategy
- Set quality thresholds for each metric
- Monitor component performance separately
- Track user satisfaction and relevance feedback
- Run A/B tests for improvement validation